# Reconstructing the Fama-French Factors from Scratch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/convexpi/lab/blob/main/notebooks/factor_reconstruction.ipynb)

This notebook **recomputes** the value (HML), size (SMB), and momentum (WML) factors from their
underlying building-block portfolios — exactly how Fama & French construct them — then checks the
reconstruction against the published factors and runs the out-of-sample (McLean & Pontiff) test.

Unlike the in-browser playground (which ships a static snapshot), this pulls the **full, current**
Ken-French data live.

In [ ]:
!pip -q install pandas-datareader

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from pandas_datareader import data as pdr

# 6 portfolios on size x book-to-market, and size x prior-return — the building blocks.
bm  = pdr.DataReader("6_Portfolios_2x3_daily", "famafrench", start="1926-01-01")[0] / 100
mom = pdr.DataReader("6_Portfolios_ME_Prior_12_2_daily", "famafrench", start="1926-01-01")[0] / 100
ff  = pdr.DataReader("F-F_Research_Data_Factors_daily", "famafrench", start="1926-01-01")[0] / 100
wm  = pdr.DataReader("F-F_Momentum_Factor_daily", "famafrench", start="1926-01-01")[0] / 100
bm.columns = [c.strip() for c in bm.columns]; mom.columns = [c.strip() for c in mom.columns]
print(bm.columns.tolist()); print(mom.columns.tolist())

## Reconstruct the factors

- **HML** = avg(value corners) − avg(growth corners)
- **SMB** = avg(small portfolios) − avg(big portfolios)
- **WML** = avg(winner corners) − avg(loser corners)

In [ ]:
HML = 0.5*(bm["SMALL HiBM"] + bm["BIG HiBM"]) - 0.5*(bm["SMALL LoBM"] + bm["BIG LoBM"])
SMB = bm[["SMALL LoBM","ME1 BM2","SMALL HiBM"]].mean(1) - bm[["BIG LoBM","ME2 BM2","BIG HiBM"]].mean(1)
WML = 0.5*(mom["SMALL HiPRIOR"] + mom["BIG HiPRIOR"]) - 0.5*(mom["SMALL LoPRIOR"] + mom["BIG LoPRIOR"])

for name, recon, pub in [("HML", HML, ff["HML"]), ("SMB", SMB, ff["SMB"]), ("WML", WML, wm.iloc[:,0])]:
    j = pd.concat([recon, pub], axis=1).dropna()
    print(f"{name}: correlation with published = {j.iloc[:,0].corr(j.iloc[:,1]):.4f}")

## Out-of-sample test

In [ ]:
sharpe = lambda x: np.sqrt(252) * x.mean() / x.std()
for name, r, py in [("HML", HML, 1993), ("SMB", SMB, 1981), ("WML", WML, 1993)]:
    r = r.dropna(); yrs = r.index.year
    ins, oos = r[yrs < py], r[yrs >= py]
    print(f"{name:4} pub {py}:  IS {sharpe(ins):+.2f}   OOS {sharpe(oos):+.2f}   decay {1-sharpe(oos)/sharpe(ins):.0%}")

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, (name, r, py) in zip(ax, [("HML",HML,1993),("SMB",SMB,1981),("WML",WML,1993)]):
    (1+r.dropna()).cumprod().plot(ax=a, logy=True, title=f"{name} (recomputed)")
    a.axvline(pd.Timestamp(f"{py}-01-01"), color="crimson", ls="--")
plt.tight_layout(); plt.show()

## Your turn
Change the publication-year split, add transaction costs, or build a different factor from the
component portfolios. This is a *real* reconstruction — every number traces back to tradeable
building blocks.